# Datasaurus – Results & Visualizations

Clean collection of all plots for the Algorithm Engineering paper.  
Each plot shows the transformation result alongside statistics comparison.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
import os
import glob

# Consistent style for all plots
plt.rcParams.update({
    'font.family': 'serif',
    'axes.titlesize': 16,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.dpi': 150,
})
mpl.rcParams['animation.embed_limit'] = 200.0

DATA_DIR = '../data'
FRAME_DIR = '../frames'

def get_stats(df):
    """Calculate dataset statistics matching the C++ implementation (Bessel correction)."""
    return {
        'mean_x': df[0].mean(),
        'mean_y': df[1].mean(),
        'std_x':  df[0].std(),      # pandas uses ddof=1 by default (Bessel)
        'std_y':  df[1].std(),
        'corr':   df[[0,1]].corr().iloc[0, 1]
    }

def stats_table(s1, s2, label1='Baseline', label2='Result'):
    """Return a formatted statistics comparison string."""
    lines = []
    lines.append(f"{'Metric':<12} {'':>2} {label1:>12}   {label2:>12}   {'Delta':>10}")
    lines.append('-' * 58)
    for key, name in [('mean_x','Mean X'), ('mean_y','Mean Y'),
                       ('std_x','Std X'), ('std_y','Std Y'), ('corr','Corr')]:
        d = abs(s1[key] - s2[key])
        lines.append(f"{name:<12}    {s1[key]:>12.6f}   {s2[key]:>12.6f}   {d:>10.2e}")
    return '\n'.join(lines)

print('Setup complete.')

## 1. Side-by-Side Comparisons (All Targets)

For each available target shape, show **Baseline (Dino)** vs. **Result** with a statistics table below.

In [ ]:
available_targets = sorted([d for d in os.listdir(FRAME_DIR)
                            if os.path.isdir(os.path.join(FRAME_DIR, d))])

for target in available_targets:
    frame_dir = os.path.join(FRAME_DIR, target)
    files = sorted(glob.glob(os.path.join(frame_dir, 'frame_*.csv')))
    if len(files) < 2:
        continue

    # Load first frame (baseline) and final result
    df_start = pd.read_csv(files[0], header=None)
    final_path = os.path.join(frame_dir, 'frame_final.csv')
    df_end = pd.read_csv(final_path if os.path.exists(final_path) else files[-1], header=None)

    s = get_stats(df_start)
    e = get_stats(df_end)
    target_name = target.replace('_', ' ').title()
    point_size = 2 if len(df_start) > 500 else 10

    # Create figure
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
    plt.subplots_adjust(bottom=0.32)

    ax1.scatter(df_start[0], df_start[1], s=point_size, color='#555555', alpha=0.6, edgecolors='none')
    ax1.set_title('Baseline (Dinosaur)')

    ax2.scatter(df_end[0], df_end[1], s=point_size, color='#c0392b', alpha=0.7, edgecolors='none')
    ax2.set_title(f'Result ({target_name})')

    for ax in [ax1, ax2]:
        ax.set_xlim(0, 100)
        ax.set_ylim(0, 100)
        ax.set_aspect('equal')
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.grid(True, linestyle='--', alpha=0.2)

    # Statistics table below the plots
    table_text = stats_table(s, e, 'Baseline', target_name)
    fig.text(0.5, 0.02, table_text, ha='center', fontsize=9, family='monospace',
             bbox=dict(boxstyle='round', facecolor='#f8f8f8', edgecolor='#cccccc'))

    fig.suptitle(f'Datasaurus: Dino → {target_name}', fontsize=18, fontweight='bold', y=0.98)
    plt.show()

## 2. Statistics Stability Over Time

Track how the five statistics remain stable across the entire optimization for each target.

In [ ]:
for target in available_targets:
    frame_dir = os.path.join(FRAME_DIR, target)
    files = sorted(glob.glob(os.path.join(frame_dir, 'frame_*.csv')))
    # Skip final, only use numbered frames
    numbered = [f for f in files if 'final' not in f]
    if len(numbered) < 3:
        continue

    iterations = []
    stats_over_time = {k: [] for k in ['mean_x', 'mean_y', 'std_x', 'std_y', 'corr']}

    for f in numbered:
        base = os.path.basename(f).replace('frame_', '').replace('.csv', '')
        try:
            it = int(base)
        except ValueError:
            continue
        df = pd.read_csv(f, header=None)
        st = get_stats(df)
        iterations.append(it)
        for k in stats_over_time:
            stats_over_time[k].append(st[k])

    target_name = target.replace('_', ' ').title()

    fig, axes = plt.subplots(2, 3, figsize=(15, 7))
    axes = axes.flatten()
    colors = ['#2c3e50', '#e74c3c', '#2980b9', '#27ae60', '#8e44ad']
    labels = ['Mean X', 'Mean Y', 'Std X', 'Std Y', 'Correlation']

    for i, (key, label) in enumerate(zip(stats_over_time.keys(), labels)):
        axes[i].plot(iterations, stats_over_time[key], color=colors[i], linewidth=1)
        axes[i].set_title(label)
        axes[i].set_xlabel('Iteration')
        axes[i].ticklabel_format(style='sci', axis='x', scilimits=(0,0))
        axes[i].grid(True, alpha=0.2)

    axes[5].axis('off')  # Hide unused 6th subplot
    fig.suptitle(f'Statistics Stability: {target_name}', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 3. Transformation Animations

Animated videos of the optimization process for each target shape. Uses adaptive frame selection:
- First 150 snapshots in full detail (early action phase)
- Remaining frames subsampled every 10th (polishing phase)
- Final result at the end

In [ ]:
for target in available_targets:
    frame_dir = os.path.join(FRAME_DIR, target)
    all_files = sorted(glob.glob(os.path.join(frame_dir, 'frame_*.csv')))
    if len(all_files) < 5:
        continue

    target_name = target.replace('_', ' ').title()

    # Adaptive frame selection
    action_phase = all_files[:150]
    polishing_phase = all_files[150::10]
    final_file = [f for f in all_files if 'final' in f]
    frames = action_phase + polishing_phase + final_file

    # Load baseline stats from first frame
    df_base = pd.read_csv(all_files[0], header=None)
    base_stats = get_stats(df_base)
    point_size = 1.5 if len(df_base) > 500 else 8

    fig, ax = plt.subplots(figsize=(8, 8))
    scatter = ax.scatter([], [], s=point_size, color='#2c3e50', alpha=0.7, edgecolors='none')
    stats_text = ax.text(0.03, 0.97, '', transform=ax.transAxes, fontsize=9,
                         verticalalignment='top', family='monospace',
                         bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

    def make_init(ax_ref, name):
        def init():
            ax_ref.set_xlim(0, 100)
            ax_ref.set_ylim(0, 100)
            ax_ref.set_aspect('equal')
            ax_ref.set_title(f'Dino → {name}', fontsize=14)
            ax_ref.set_xlabel('X')
            ax_ref.set_ylabel('Y')
            return scatter, stats_text
        return init

    def make_update(scatter_ref, text_ref, base_s):
        def update(frame_path):
            df = pd.read_csv(frame_path, header=None)
            scatter_ref.set_offsets(np.c_[df[0], df[1]])

            mx, my = df[0].mean(), df[1].mean()
            sx, sy = df[0].std(), df[1].std()
            corr = df[[0,1]].corr().iloc[0, 1]

            base = os.path.basename(frame_path).replace('frame_', '').replace('.csv', '')
            if base == 'final':
                iter_label = 'FINAL'
            else:
                try:
                    iter_label = f'{int(base):,}'
                except ValueError:
                    iter_label = 'N/A'

            info = (f'Iteration: {iter_label}\n'
                    f'Points:    {len(df)}\n'
                    f'{"-"*26}\n'
                    f'Mean X: {mx:8.4f}  (base: {base_s["mean_x"]:.4f})\n'
                    f'Mean Y: {my:8.4f}  (base: {base_s["mean_y"]:.4f})\n'
                    f'Std X:  {sx:8.4f}  (base: {base_s["std_x"]:.4f})\n'
                    f'Std Y:  {sy:8.4f}  (base: {base_s["std_y"]:.4f})\n'
                    f'Corr:   {corr:8.4f}  (base: {base_s["corr"]:.4f})')
            text_ref.set_text(info)
            return scatter_ref, text_ref
        return update

    ani = FuncAnimation(fig,
                        make_update(scatter, stats_text, base_stats),
                        frames=frames,
                        init_func=make_init(ax, target_name),
                        blit=True, interval=50)

    plt.close(fig)
    display(HTML(f'<h3>Animation: Dino → {target_name} ({len(frames)} frames)</h3>'))
    display(HTML(ani.to_jshtml()))

## 4. Export Comparison Plots (PNG, 300 DPI)

Saves publication-ready PNGs for each transformation to the Notebook directory.

In [ ]:
for target in available_targets:
    frame_dir = os.path.join(FRAME_DIR, target)
    files = sorted(glob.glob(os.path.join(frame_dir, 'frame_*.csv')))
    if len(files) < 2:
        continue

    df_start = pd.read_csv(files[0], header=None)
    final_path = os.path.join(frame_dir, 'frame_final.csv')
    df_end = pd.read_csv(final_path if os.path.exists(final_path) else files[-1], header=None)

    s = get_stats(df_start)
    e = get_stats(df_end)
    target_name = target.replace('_', ' ').title()
    point_size = 2 if len(df_start) > 500 else 10

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
    plt.subplots_adjust(bottom=0.32)

    ax1.scatter(df_start[0], df_start[1], s=point_size, color='#555555', alpha=0.6, edgecolors='none')
    ax1.set_title('Baseline (Dinosaur)')

    ax2.scatter(df_end[0], df_end[1], s=point_size, color='#c0392b', alpha=0.7, edgecolors='none')
    ax2.set_title(f'Result ({target_name})')

    for ax in [ax1, ax2]:
        ax.set_xlim(0, 100)
        ax.set_ylim(0, 100)
        ax.set_aspect('equal')
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.grid(True, linestyle='--', alpha=0.2)

    table_text = stats_table(s, e, 'Baseline', target_name)
    fig.text(0.5, 0.02, table_text, ha='center', fontsize=9, family='monospace',
             bbox=dict(boxstyle='round', facecolor='#f8f8f8', edgecolor='#cccccc'))

    fig.suptitle(f'Datasaurus: Dino → {target_name}', fontsize=18, fontweight='bold', y=0.98)

    out_path = f'comparison_{target}.png'
    plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f'Saved: {out_path}')